# STAC data prep — HydroRIVERS (Africa) (GeoParquet)

Converts the HydroRIVERS (Africa) shapefile to a single **GeoParquet** file and writes the enriched metadata next to it.

- **Case:** vector data, one layer, 1.5 million river reaches (lines) with 14 attribute columns, no time dimension (~226 MB .shp + 150 MB .dbf as shapefile). Source CRS: WGS84 (EPSG:4326).
- **Format:** one GeoParquet (see `DATA_FORMATS.md`, vector row; same approach as `18_hydrolakes` and `19_country_boundaries`). Rows are sorted along a Hilbert curve and a bbox covering column is written, so readers can filter spatially without loading everything.
- **OGC service:** not created. The GeoServer of the Global Coastal Atlas has no GeoParquet data store, and the shapefile is not published instead.
- **Metadata template:** `metadata/metadata_HydroRIVERS.json`. It is incomplete (empty bbox, time, keywords, media type, collection id), so the missing fields are filled below.

Source: `P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\HydroRIVERS_v10_af.shp`

**Running in docker:** the `P:` drive is not visible in the container, so run `docker\run_notebook_with_p_drive.ps1 -Notebook 25_hydrorivers.ipynb -Variable HydroRIVERS_v10_af` from the host. It copies the shapefile parts to a local mirror, executes this notebook, copies `stac\` back to `P:`, verifies the hashes and removes the mirror. The notebook needs about 4 GB of memory.

### Import packages

In [1]:
import json
import os
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

### Define drive paths

`DATA_VIVIAN` and `STAC_DATA_REPO` are set in the docker container; otherwise the Windows host paths are used.

In [2]:
VARIABLE = "HydroRIVERS_v10_af"
COLLECTION_ID = "hydrorivers"
PARQUET_NAME = "hydrorivers_af"

# DATA_VIVIAN is set in docker, where the P: drive is not mounted (a local mirror is used and stac/ is copied back to P:)
data_vivian = Path(os.environ.get("DATA_VIVIAN", r"P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian"))
stac_repo = Path(os.environ.get("STAC_DATA_REPO", r"C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\global-coastal-atlas\STAC\data"))

source_shp = data_vivian / f"{VARIABLE}.shp"
metadata_template_path = stac_repo / "metadata" / "metadata_HydroRIVERS.json"

processed_data_dir = data_vivian / "stac" / COLLECTION_ID
parquet_dir = processed_data_dir / "parquet"
parquet_dir.mkdir(parents=True, exist_ok=True)

for p in (source_shp, metadata_template_path):
    if not p.is_file():
        raise FileNotFoundError(p)
print("Source:", source_shp)
print("Metadata template:", metadata_template_path)
print("Output dir:", processed_data_dir)

Source: P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\HydroRIVERS_v10_af.shp
Metadata template: C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\global-coastal-atlas\STAC\data\metadata\metadata_HydroRIVERS.json
Output dir: P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\stac\hydrorivers


### Read raw data

Takes a few minutes and about 4 GB of memory.

In [3]:
gdf = gpd.read_file(source_shp, engine="pyogrio", encoding="utf-8")

empty = gdf.geometry.isna() | gdf.geometry.is_empty
print("rows:", len(gdf), "| crs:", gdf.crs.to_string() if gdf.crs else None)
print("geometry types:", gdf.geom_type.value_counts().to_dict())
print("HYRIV_ID unique:", gdf["HYRIV_ID"].is_unique, "| null or empty geometries:", int(empty.sum()))
print("total bounds:", gdf.total_bounds.tolist())
gdf.drop(columns="geometry").head()

rows: 1527551 | crs: EPSG:4326
geometry types: {'LineString': 1527551}
HYRIV_ID unique: True | null or empty geometries: 0
total bounds: [-17.99374999999995, -34.79374999999997, 54.41666666666555, 37.331249999998924]


,HYRIV_ID,NEXT_DOWN,MAIN_RIV,LENGTH_KM,DIST_DN_KM,DIST_UP_KM,CATCH_SKM,UPLAND_SKM,ENDORHEIC,DIS_AV_CMS,ORD_STRA,ORD_CLAS,ORD_FLOW,HYBAS_L12
0,10000001,0,10000001,0.89,0.0,7.2,11.27,11.1,0,0.062,1,1,8,1120031210
1,10000002,0,10000002,2.90,0.0,7.0,24.59,24.2,0,0.126,1,1,7,1120031210
2,10000003,10000009,10000009,4.63,5.7,9.8,57.23,57.2,0,0.316,1,1,7,1120031210
3,10000004,10000009,10000009,0.69,5.7,5.4,11.11,11.1,0,0.061,1,2,8,1120031210
4,10000005,0,10000005,8.32,0.0,13.6,35.02,34.0,0,0.177,1,1,7,1120031210


### Check the CRS

The shapefile is in WGS84, the CRS the STAC catalog uses. A missing `.prj` is set to EPSG:4326; any other CRS is reprojected.

In [4]:
print("CRS before:", gdf.crs.to_string() if gdf.crs else None)
if gdf.crs is None:
    gdf = gdf.set_crs(4326)
elif gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(4326)
print("CRS:", gdf.crs.to_string())
print("total bounds (EPSG:4326):", gdf.total_bounds.tolist())

CRS before: EPSG:4326
CRS: EPSG:4326
total bounds (EPSG:4326): [-17.99374999999995, -34.79374999999997, 54.41666666666555, 37.331249999998924]


### Make the metadata complete (dataset dependent)

Fill the fields that are empty in the template. Column descriptions follow the HydroSHEDS technical documentation v1.4 (`HydroSHEDS_TechDoc_v1_4.pdf`); please double-check the descriptions that are marked with `?` before publishing.

In [5]:
with open(metadata_template_path, encoding="utf-8") as f:
    metadata = json.load(f)

# Descriptions per attribute column (used as table:columns in the STAC item)
table_columns = {
    "HYRIV_ID": "Unique river reach identifier",
    "NEXT_DOWN": "ID of the next downstream reach (0 for reaches that end at the ocean or in an endorheic sink)",
    "MAIN_RIV": "ID of the most downstream reach of the main river the reach belongs to",
    "LENGTH_KM": "Length of the reach (km)",
    "DIST_DN_KM": "Distance from the reach outlet to the river mouth / sink (km)",
    "DIST_UP_KM": "Distance from the reach outlet to the most upstream point of the river (km)",
    "CATCH_SKM": "Catchment area of the reach itself (km2)",
    "UPLAND_SKM": "Total upstream drainage area at the reach outlet (km2)",
    "ENDORHEIC": "Endorheic flag: 0 = drains to the ocean, 1 = reach flows into an endorheic basin, 2 = reach ends in an endorheic sink (?)",
    "DIS_AV_CMS": "Average long-term discharge (m3/s)",
    "ORD_STRA": "River order, Strahler classification",
    "ORD_CLAS": "River order, classification by average discharge",
    "ORD_FLOW": "River order, classification by discharge (log10 scale)",
    "HYBAS_L12": "ID of the HydroBASINS level 12 sub-basin the reach resides in",
    "geometry": "River reach line (EPSG:4326)",
}
missing = set(gdf.columns) - set(table_columns)
assert not missing, f"Columns without description: {missing}"

west, south, east, north = (float(v) for v in gdf.total_bounds)
bbox = [max(west, -180.0), max(south, -90.0), min(east, 180.0), min(north, 90.0)]

# Extra info that the raw template does not contain
metadata.update({
    "TITLE_ABBREVIATION": "HydroRIVERS (Africa)",
    "HISTORY": [
        "HydroRIVERS version 1.0 (Lehner and Grill, 2013), Africa shapefile HydroRIVERS_v10_af.shp",
        "Converted to a single GeoParquet (zstd, EPSG:4326, rows sorted along a Hilbert curve, bbox covering column)",
        "Geometries and attributes are unchanged"
],
    "MEDIA_TYPE": "application/vnd.apache.parquet",
    "DATA_MODEL": "vector_geoparquet",
    "DIMENSIONS": [],
    "SPATIAL_EXTENT": [round(v, 6) for v in bbox],
    # Publication year; the dataset has no time dimension
    "TEMPORAL_EXTENT": ["2013-01-01T00:00:00Z", ""],
    "KEYWORDS": ["rivers", "river network", "discharge", "hydrology", "surface water", "HydroSHEDS", "Africa"],
    "TAGS": ["rivers", "lines", "africa", "geoparquet"],
    "LONG_NAME": "River network of Africa (river reaches)",
    "UNITS": "",
    "COMMENT": (
        "Vector layer with one line per river reach for Africa only (HydroRIVERS_v10_af); the other continents are "
        "not published. Rivers with a catchment area >= 10 km2 or an average flow >= 0.1 m3/s. The temporal extent is "
        "the publication year; the layer is static."
    ),
    "AUTHOR": "Lehner B, Grill G",
    "CRS": "EPSG:4326",
    "ITEM_BBOX_CRS": "EPSG:4326",
    "SPATIAL_RESOLUTION": None,
    "NODATA": None,
    "DATA_TYPE": "GeoParquet",
    "COLLECTION_ID": COLLECTION_ID,
    "WMS_DATASET": "",  # no OGC service: GeoServer has no GeoParquet data store
    "FEATURE_COUNT": int(len(gdf)),
    "TABLE_COLUMNS": table_columns,
})
metadata["DOI"] = metadata["DOI"].replace("https://doi.org/https://doi.org/", "https://doi.org/")  # doubled prefix in the template

print("SPATIAL_EXTENT:", metadata["SPATIAL_EXTENT"])
print("FEATURE_COUNT:", metadata["FEATURE_COUNT"])

SPATIAL_EXTENT: [-17.99375, -34.79375, 54.416667, 37.33125]
FEATURE_COUNT: 1527551


### Write data to GeoParquet

`zstd` compression, row groups of 100 000 features, rows ordered along a Hilbert curve so that nearby features share a row group, and a `bbox` covering column for spatial filtering. Null or empty geometries (if any) are put at the end, because they have no position on the curve.

In [6]:
valid = ~(gdf.geometry.isna() | gdf.geometry.is_empty).to_numpy()
hilbert = gdf.geometry[valid].hilbert_distance().to_numpy()
order = np.concatenate([np.flatnonzero(valid)[np.argsort(hilbert)], np.flatnonzero(~valid)])
gdf = gdf.iloc[order].reset_index(drop=True)

out_path = parquet_dir / f"{PARQUET_NAME}.parquet"
gdf.to_parquet(
    out_path,
    index=False,
    compression="zstd",
    row_group_size=100_000,
    write_covering_bbox=True,
)

metadata_output_path = processed_data_dir / f"metadata_{COLLECTION_ID}.json"
with open(metadata_output_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4, ensure_ascii=False)

before_mb = sum(p.stat().st_size for p in source_shp.parent.glob(f"{VARIABLE}.*")) / 1e6
print(f"Wrote GeoParquet: {out_path} ({out_path.stat().st_size / 1e6:.0f} MB, shapefile parts {before_mb:.0f} MB)")
print("Wrote metadata:", metadata_output_path)

Wrote GeoParquet: P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\stac\hydrorivers\parquet\hydrorivers_af.parquet (85 MB, shapefile parts 419 MB)
Wrote metadata: P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\stac\hydrorivers\metadata_hydrorivers.json


### Verify the output

Same rows (in the same order), columns and values as the processed GeoDataFrame, EPSG:4326, readable as GeoParquet.

In [7]:
meta = pq.ParquetFile(out_path).metadata
print("row groups:", meta.num_row_groups, "| rows:", meta.num_rows)
assert meta.num_rows == len(gdf)

new = gpd.read_parquet(out_path)
assert new.crs.to_epsg() == 4326
assert list(new.columns.drop("bbox", errors="ignore")) == list(gdf.columns)

# Rows were written in the order of gdf, so they can be compared position by position
attrs = [c for c in gdf.columns if c != "geometry"]
pd.testing.assert_frame_equal(pd.DataFrame(new[attrs]), pd.DataFrame(gdf[attrs]))
same_geom = new.geometry.geom_equals_exact(gdf.geometry, tolerance=0) | (new.geometry.isna() & gdf.geometry.isna())
assert same_geom.all(), "Geometries changed"
print("OK: hydrorivers GeoParquet and metadata are ready for the STAC builder notebook.")

row groups: 16 | rows: 1527551
OK: hydrorivers GeoParquet and metadata are ready for the STAC builder notebook.
